In [ ]:
from pathlib import Path

from generative_physics.config import TrainingConfig
from generative_physics.run import run_training

config = TrainingConfig(
    pde_kind="navier_stokes",
    train_image_size=256,
    output_image_size=256,
    ks_grid_size=512,
    ks_condition_encoding="y_constant",
    ks_debug_num_samples=50,
    
    num_eval_pairs=8,

    stream_chunk_size=32,

    sim_num_workers=16,
    validate_every_n_steps=500,
    validation_num_images=8,

    run_initial_validation=True,
)


In [ ]:
results = run_training(config)


LoRA steps loss=0.0469 ema=0.0526:  82%|████████▏ | 8159/10001 [1:28:20<21:22,  1.44it/s] , loss=0.0469 ema=0.0526 lr=2.44e-06

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

from generative_physics.ks import (
    KS_BURN_T,
    KS_CMAP_NAME,
    KS_DOMAIN_LENGTH,
    KS_GRID_SIZE,
    KS_HORIZON_T,
    KS_INITIAL_AMP,
    KS_INITIAL_DECAY,
    KS_INITIAL_NUM_MODES,
    KS_STEPS_PER_FRAME,
    KS_VALUE_BOUNDS,
    array_to_flame_pil,
    augment_ks_symmetry,
    flame_image_to_normalized_array,
    ks_operators,
    ks_effective_discretization,
    random_spectral_ic,
    resize_ks_scalar_render,
)

KS_CONVERGENCE_SAMPLES = 50
KS_CONVERGENCE_RESOLUTIONS = (512, 1024, 2048)
KS_CONVERGENCE_REFERENCE = 2048
KS_CONVERGENCE_SEED_OFFSET = 30_000


@torch.no_grad()
def ks_integrate_cnab2_batch(u0, *, Lx, dt, Nt, nsave):
    u0 = u0.to(torch.float64)
    batch_size, nx = u0.shape
    G, A_inv, B = ks_operators(Nx=nx, Lx=Lx, dt=dt, device=u0.device)
    uhat = torch.fft.fft(u0, dim=-1)
    uhat[..., 0] = 0.0
    uhat[..., nx // 2] = 0.0

    def nonlinear(current_uhat):
        u = torch.fft.ifft(current_uhat, dim=-1).real
        value = G * torch.fft.fft(u * u, dim=-1)
        value[..., 0] = 0.0
        value[..., nx // 2] = 0.0
        return value

    nonlinear_previous = nonlinear(uhat)
    nonlinear_current = nonlinear_previous.clone()
    trajectory = torch.empty(
        (batch_size, Nt // nsave + 1, nx),
        device=u0.device,
        dtype=torch.float64,
    )
    trajectory[:, 0] = torch.fft.ifft(uhat, dim=-1).real
    output_index = 1
    for step in range(1, Nt + 1):
        uhat = A_inv * (
            B * uhat
            + 1.5 * dt * nonlinear_current
            - 0.5 * dt * nonlinear_previous
        )
        uhat[..., 0] = 0.0
        uhat[..., nx // 2] = 0.0
        nonlinear_previous = nonlinear_current
        nonlinear_current = nonlinear(uhat)
        if step % nsave == 0:
            trajectory[:, output_index] = torch.fft.ifft(uhat, dim=-1).real
            output_index += 1
    return trajectory


def decode_rendered_trajectories(trajectories, image_size):
    decoded = np.empty(
        (len(trajectories), image_size, image_size), dtype=np.float32
    )
    for sample_index, trajectory in enumerate(
        tqdm(trajectories, desc="decoding trajectories", leave=False)
    ):
        rendered = resize_ks_scalar_render(trajectory, image_size)
        image = array_to_flame_pil(
            rendered,
            value_bounds=KS_VALUE_BOUNDS,
            cmap_name=KS_CMAP_NAME,
        )
        decoded[sample_index] = 1.0 + flame_image_to_normalized_array(
            image, cmap_name=KS_CMAP_NAME
        )
    return decoded


simulation_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
generators = []
spectral_initial_conditions = []
for sample_index in range(KS_CONVERGENCE_SAMPLES):
    generator = torch.Generator(device=simulation_device).manual_seed(
        KS_CONVERGENCE_SEED_OFFSET + sample_index
    )
    generators.append(generator)
    spectral_initial_conditions.append(
        random_spectral_ic(
            Nx=KS_CONVERGENCE_REFERENCE,
            n_modes=KS_INITIAL_NUM_MODES,
            decay=KS_INITIAL_DECAY,
            amp=KS_INITIAL_AMP,
            generator=generator,
            device=simulation_device,
        )
    )
spectral_initial_conditions = torch.stack(spectral_initial_conditions)

_, _, reference_dt = ks_effective_discretization(
    Nx=KS_GRID_SIZE, image_size=KS_CONVERGENCE_REFERENCE
)
burn_steps = int(round(KS_BURN_T / reference_dt))
burned = ks_integrate_cnab2_batch(
    spectral_initial_conditions,
    Lx=KS_DOMAIN_LENGTH,
    dt=reference_dt,
    Nt=burn_steps,
    nsave=burn_steps,
)[:, -1]
initial_reference = torch.stack(
    [augment_ks_symmetry(u, generator=g) for u, g in zip(burned, generators)]
)

reference_trajectory = ks_integrate_cnab2_batch(
    initial_reference,
    Lx=KS_DOMAIN_LENGTH,
    dt=reference_dt,
    Nt=(KS_CONVERGENCE_REFERENCE - 1) * KS_STEPS_PER_FRAME,
    nsave=KS_STEPS_PER_FRAME,
).cpu().numpy()
reference_decoded = decode_rendered_trajectories(
    reference_trajectory, KS_CONVERGENCE_REFERENCE
)

mean_relative_error = {}
for resolution in (512, 1024):
    _, _, resolution_dt = ks_effective_discretization(
        Nx=KS_GRID_SIZE, image_size=resolution
    )
    initial = initial_reference[:, :: KS_CONVERGENCE_REFERENCE // resolution].contiguous()
    trajectory = ks_integrate_cnab2_batch(
        initial,
        Lx=KS_DOMAIN_LENGTH,
        dt=resolution_dt,
        Nt=(resolution - 1) * KS_STEPS_PER_FRAME,
        nsave=KS_STEPS_PER_FRAME,
    ).cpu().numpy()
    decoded = decode_rendered_trajectories(trajectory, resolution)
    reference_on_grid = F.interpolate(
        torch.from_numpy(reference_decoded)[:, None],
        size=(resolution, resolution),
        mode="bilinear",
        align_corners=True,
    )[:, 0].numpy()
    numerator = np.linalg.norm(decoded - reference_on_grid, axis=-1)
    denominator = np.linalg.norm(reference_on_grid, axis=-1)
    relative_error = numerator / np.maximum(denominator, np.finfo(np.float32).eps)
    mean_relative_error[resolution] = relative_error.mean(axis=0)
    del trajectory, decoded, reference_on_grid, relative_error

fig, ax = plt.subplots(figsize=(8, 3.5))
for resolution in (512, 1024):
    times = np.linspace(0.0, KS_HORIZON_T, resolution)
    ax.plot(
        times,
        mean_relative_error[resolution],
        linewidth=2.0,
        label=f"{resolution} vs 1024",
    )
ax.set_xlabel("time")
ax.set_ylabel(r"$\mathbb{E}_u[\|u-u_{ref}\|_2/\|u_{ref}\|_2]$")
ax.set_title("KS resolution error at fixed L and T from 50 initial conditions")
ax.grid(True, alpha=0.25)
ax.legend(frameon=False)
plt.tight_layout()
plt.show()
